# Brain MRI Tumor Classification - Exploratory Data Analysis (EDA)

This notebook performs comprehensive Exploratory Data Analysis on the Brain MRI dataset. It covers:
- Dataset overview and counts
- Missing or corrupted file detection
- Class distribution analysis and imbalance visualization
- Resolution and aspect ratio analysis
- Pixel intensity distributions
- Visual examination of raw scans

In [ ]:
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image

# Append project root to path for local imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root added to path: {project_root}")

In [ ]:
from src import config
from src.utils import setup_logger, set_seed
from src.data.dataset_loader import MRIDatasetLoader

# Setup logging and configurations
setup_logger(log_dir=str(project_root / "logs"))
set_seed(config.SEED)

print(f"Configured dataset path: {config.DATASET_PATH}")

## 1. Load Dataset Metadata

We scan the directories `dataset/train`, `dataset/validation`, and `dataset/test` for scans and build metadata.

In [ ]:
loader = MRIDatasetLoader(config.DATASET_PATH)
metadata = loader.scan_dataset()

# Determine if the dataset contains images
has_images = any(df is not None and not df.empty for df in metadata.values())

if not has_images:
    print("\n" + "!"*60)
    print("WARNING: Dataset directory is currently empty or structure is missing.")
    print(f"Please download the MRI dataset and place it under: {config.DATASET_PATH}")
    print("Make sure it has the splits: train/, validation/, test/")
    print("And class subfolders: glioma/, meningioma/, pituitary/, notumor/")
    print("!"*60 + "\n")
else:
    loader.print_summary()

## 2. Missing/Corrupted Image Report

Check if any image file failed integrity verification or is otherwise unreadable.

In [ ]:
print("=== CORRUPTED IMAGE REPORT ===")
corrupted_found = False
for split, paths in loader.corrupted_files.items():
    print(f"Split '{split}': {len(paths)} corrupted or non-image files skipped.")
    if paths:
        corrupted_found = True
        for path in paths[:5]:
            print(f"  - {path}")
        if len(paths) > 5:
            print(f"  - ... and {len(paths) - 5} more.")

if not corrupted_found:
    print("No corrupted images discovered in any split!")

## 3. Class Distribution & Imbalance Visualization

Visualizing class distributions helps diagnose class imbalance which could bias classification performance.

In [ ]:
def plot_class_distributions(loader):
    dfs = []
    for split, df in loader.metadata.items():
        if df is not None and not df.empty:
            df_copy = df.copy()
            df_copy['split'] = split
            dfs.append(df_copy)
    
    if not dfs:
        print("No data available to plot class distributions.")
        return
        
    all_data = pd.concat(dfs, ignore_index=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Bar Chart of counts by split
    sns.countplot(data=all_data, x='class', hue='split', ax=axes[0], order=loader.classes, palette='viridis')
    axes[0].set_title("Class Distribution across Train/Validation/Test Splits")
    axes[0].set_xlabel("Tumor Classification Class")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)
    
    # 2. Pie Chart overall
    overall_counts = all_data['class'].value_counts()
    axes[1].pie(
        overall_counts, 
        labels=overall_counts.index, 
        autopct='%1.1f%%', 
        startangle=90, 
        colors=sns.color_palette('pastel'),
        wedgeprops={'edgecolor': 'white', 'linewidth': 1}
    )
    axes[1].set_title("Overall Dataset Class Representation")
    
    plt.tight_layout()
    plt.show()

if has_images:
    plot_class_distributions(loader)
else:
    print("Skipping plot: No images loaded.")

## 4. Image Resolution & Dimension Analysis

Verifying image dimensions shows if scans are uniform or if high variability in aspect ratios is present.

In [ ]:
def plot_resolutions(loader):
    dfs = [df for df in loader.metadata.values() if df is not None and not df.empty]
    if not dfs:
        print("No data available to plot resolutions.")
        return
    all_data = pd.concat(dfs, ignore_index=True)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=all_data, x='width', y='height', hue='class', style='class', s=70, alpha=0.7, palette='Set2')
    plt.title("Image Dimensions Scatter (Width vs Height)")
    plt.xlabel("Width (Pixels)")
    plt.ylabel("Height (Pixels)")
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Add target resolution line from config
    target_h, target_w = config.IMAGE_SIZE
    plt.axvline(x=target_w, color='r', linestyle=':', label=f'Target Width ({target_w})')
    plt.axhline(y=target_h, color='r', linestyle=':', label=f'Target Height ({target_h})')
    plt.legend()
    plt.show()

if has_images:
    plot_resolutions(loader)
else:
    print("Skipping plot: No images loaded.")

## 5. Pixel Intensity Distribution

Shows density curves of average grayscale intensities. This is useful to observe if scanner intensities differ substantially by tumor class.

In [ ]:
def plot_pixel_intensities(loader):
    dfs = [df for df in loader.metadata.values() if df is not None and not df.empty]
    if not dfs:
        print("No data available to plot intensities.")
        return
    all_data = pd.concat(dfs, ignore_index=True)
    
    plt.figure(figsize=(12, 6))
    sns.kdeplot(
        data=all_data, 
        x='mean_intensity', 
        hue='class', 
        fill=True, 
        common_norm=False, 
        alpha=0.3, 
        palette='Set1'
    )
    plt.title("Distribution of Mean Pixel Intensity (Scaled [0,1]) by Class")
    plt.xlabel("Mean Pixel Intensity")
    plt.ylabel("Density")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

if has_images:
    plot_pixel_intensities(loader)
else:
    print("Skipping plot: No images loaded.")

## 6. Random Scan Viewer & Class Samples

Displaying individual scans and matrices categorized by classes ensures raw image inspection.

In [ ]:
def display_random_image(loader):
    valid_splits = [s for s, df in loader.metadata.items() if df is not None and not df.empty]
    if not valid_splits:
        print("No images available to display.")
        return
    
    split = random.choice(valid_splits)
    df = loader.metadata[split]
    row = df.sample(1).iloc[0]
    
    img = cv2.imread(row['file_path'], cv2.IMREAD_GRAYSCALE)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap='gray')
    plt.title(f"Random Viewer | Split: {split.upper()} | Class: {row['class'].upper()}\nSize: {row['width']}x{row['height']} | Intensity Mean: {row['mean_intensity']:.3f}")
    plt.axis('off')
    plt.show()

if has_images:
    display_random_image(loader)
else:
    print("Skipping display: No images loaded.")

In [ ]:
def plot_sample_grid(loader, num_samples=4):
    if not loader.classes:
        print("No classes available.")
        return
        
    dfs = [df for df in loader.metadata.values() if df is not None and not df.empty]
    if not dfs:
        print("No images to load grid.")
        return
    df = pd.concat(dfs, ignore_index=True)
    
    n_classes = len(loader.classes)
    fig, axes = plt.subplots(n_classes, num_samples, figsize=(4 * num_samples, 4 * n_classes))
    
    for i, class_name in enumerate(loader.classes):
        class_df = df[df['class'] == class_name]
        if class_df.empty:
            continue
        samples = class_df.sample(min(num_samples, len(class_df)), random_state=config.SEED)
        
        for j in range(num_samples):
            ax = axes[i, j] if n_classes > 1 else axes[j]
            if j < len(samples):
                row = samples.iloc[j]
                img = cv2.imread(row['file_path'], cv2.IMREAD_GRAYSCALE)
                ax.imshow(img, cmap='gray')
                ax.set_title(f"{class_name.upper()}\n{row['width']}x{row['height']}", fontsize=10)
            else:
                ax.text(0.5, 0.5, "No Image", ha='center', va='center')
            ax.axis('off')
            
    plt.suptitle("Brain MRI Sample Grid by Diagnosis Class", y=0.99, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if has_images:
    plot_sample_grid(loader)
else:
    print("Skipping plot: No images loaded.")